## Q4 — Age Calculation
Age is defined as the number of complete years elapsed between the driver's date of 
birth and the race date:

    age = floor(datediff(race_date, dob) / 365.25)

`datediff` returns the exact number of days between the two dates. Dividing by 365.25 
accounts for leap years (1 extra day every 4 years on average). `floor` ensures we 
count only fully completed years — e.g., a driver born on July 15, 1990 racing on 
June 1, 2010 is 19, not 20, because their birthday hasn't occurred yet that year.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_races   = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",   header=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:

df_races_clean = df_races.select(
    "raceId", "year",
    F.col("name").alias("race_name"),
    F.to_date(F.col("date"), "yyyy-MM-dd").alias("race_date")
)


In [0]:
#Driver code
df_drivers_clean = df_drivers.select(
    "driverId",
    F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("driver_name"),
    F.when(
        F.col("code").isNull() |
        (F.trim(F.col("code")) == "") |
        (F.col("code") == "\\N"),
        F.upper(F.substring(F.col("surname"), 1, 3))
    ).otherwise(F.col("code")).alias("driver_code"),
    F.to_date(F.col("dob"), "yyyy-MM-dd").alias("dob")
)

In [0]:
df_results_clean = df_results.select("raceId", "driverId")

In [0]:
df_joined = (
    df_results_clean
    .join(df_races_clean,   on="raceId",   how="left")
    .join(df_drivers_clean, on="driverId", how="left")
)

In [0]:
df_joined = df_joined.withColumn(
    "age",
    F.floor(F.datediff(F.col("race_date"), F.col("dob")) / 365.25)
)


In [0]:

window_race = Window.partitionBy("raceId")

df_q4 = (
    df_joined
    .withColumn("min_age", F.min("age").over(window_race))
    .withColumn("max_age", F.max("age").over(window_race))
    .filter(
        (F.col("age") == F.col("min_age")) | (F.col("age") == F.col("max_age"))
    )
    .withColumn("category",
        F.when(F.col("age") == F.col("min_age"), "Youngest")
         .when(F.col("age") == F.col("max_age"), "Oldest")
    )
    .dropDuplicates(["raceId", "driverId", "category"])
    .select("year", "race_name", "race_date", "category",
            "driver_code", "driver_name", "dob", "age")
    .orderBy("year", "race_name", "category")
)


In [0]:
display(df_q4)